# Session 10 · Logistic Regression I

**Machine Learning Foundations · Sanketana School of Code**

KNN **voted** — a hard "spam" or "not spam." But it never told us *how sure* it was. Today we build a classifier that answers a richer question: not just *"spam or not?"* but *"how **likely** is this spam?"* — a **probability** between 0 and 1. That model is **logistic regression**.

By the end of this notebook you will be able to:

- explain why a **straight line** can't answer a yes/no question
- read the **S-curve** that logistic regression uses to stay between 0 and 1
- get a **probability** with `predict_proba` and a yes/no with `predict`
- score the classifier with **accuracy** on a held-out test set

## Warm-up · Last session's homework

Your coach will walk through Session 9's KNN spam filter (about 10 minutes) — everyone picked a k and reported test accuracy.

KNN gave a confident vote, but *no sense of how sure it was*. A message on the knife's edge got the same "spam!" as an obvious one. Today we fix exactly that.

## Step 1 · Try the tool we already have — and watch it fail

We know how to fit a **straight line** (Module 2). So let's try it on a yes/no label: predict `is_spam` (only ever 0 or 1) from the number of **exclamation marks** in a message. Watch what the line does at the edges.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

spam = pd.read_csv("../../../datasets/secondary/spam_features.csv")
print("messages:", len(spam), " | spam rate:", round(spam["is_spam"].mean(), 3))

# One feature so we can see everything on a 2-D plot.
feat = "num_exclamation_marks"
x = spam[[feat]].values
y = spam["is_spam"].values

line = LinearRegression().fit(x, y)
grid = np.linspace(x.min(), x.max(), 100).reshape(-1, 1)
print("straight-line predictions across the range:")
print("  at fewest marks:", round(float(line.predict(grid)[0]), 2))
print("  at most marks:  ", round(float(line.predict(grid)[-1]), 2), "  <-- above 1: nonsense as a probability")

A prediction above **1** (or below **0**) can't be a probability. **A straight line refuses to stay in the [0, 1] corridor** — it's the wrong shape of tool for a yes/no question.

## Step 2 · Bend the line into an S

The fix: take that same line and pass its output through an **S-shaped curve** that squashes *any* number into 0–1. That curve is **logistic regression**. Let's fit it to the same single feature and plot both.

In [ ]:
logit = LogisticRegression().fit(x, y)
s_curve = logit.predict_proba(grid)[:, 1]        # P(spam) along the range

plt.figure(figsize=(8, 5))
# jitter the 0/1 dots vertically a touch so we can see them
jit = y + np.random.default_rng(0).uniform(-0.03, 0.03, size=len(y))
plt.scatter(x, jit, s=10, alpha=0.3, label="messages (0 = ham, 1 = spam)")
plt.plot(grid, line.predict(grid), "r--", lw=2, label="straight line (leaves 0–1!)")
plt.plot(grid, s_curve, "b-", lw=2.5, label="logistic S-curve (stays in 0–1)")
plt.axhline(0, color="gray", lw=0.8); plt.axhline(1, color="gray", lw=0.8)
plt.axhline(0.5, color="green", ls=":", lw=1.2, label="0.5 = default decision cut")
plt.xlabel("number of exclamation marks"); plt.ylabel("P(spam)")
plt.title("A straight line vs the logistic S-curve")
plt.legend(); plt.ylim(-0.3, 1.3); plt.tight_layout(); plt.show()

print("S-curve P(spam) across the range:", np.round(logit.predict_proba(
    np.linspace(0, 4, 5).reshape(-1, 1))[:, 1], 2))

**Read it.** The red line runs straight out of the top of the chart — useless as a probability. The blue S-curve hugs 0 for few marks, rises through the middle, and flattens toward 1 — a valid probability **everywhere**. That is the whole idea: *a line taught to stay between 0 and 1.*

## Step 3 · The real classifier: read the probability

Now fit logistic regression on **all** the spam features and read `predict_proba` — the model's *confidence* that each message is spam. (We scale first, as usual — established practice.)

In [ ]:
feature_cols = [c for c in spam.columns if c != "is_spam"]
X = spam[feature_cols].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# ✏️ TODO: fit logistic regression on the scaled training data.
model = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)

# predict_proba gives P(spam) for each test message; column 1 = P(class 1)
probs = model.predict_proba(X_test_s)[:, 1]
print("most spammy message   P(spam) =", round(float(probs.max()), 3))
print("least spammy message  P(spam) =", round(float(probs.min()), 3))
print("most UNSURE message    P(spam) =", round(float(probs[np.argmin(np.abs(probs - 0.5))]), 3),
      "  <-- closest to 0.5")

## Step 4 · From probability to a decision

`predict` turns the probability into a yes/no by **cutting at 0.5**: at or above 0.5 → spam, below → not. Let's confirm that, and score the classifier with **accuracy** on the held-out test set.

In [ ]:
labels = model.predict(X_test_s)

# predict is just predict_proba thresholded at 0.5 — check it:
same = np.array_equal(labels, (probs >= 0.5).astype(int))
print("predict == (probability >= 0.5)?", same)
print("test accuracy:", round(model.score(X_test_s, y_test), 3))

### ✏️ Your turn

Find the test message the model is **least sure** about (its probability is closest to 0.5). In one or two sentences: why might a spam filter treat *that* message differently from one it scored 0.99? (You don't need to solve it — just notice the question. Session 11 answers it.)

*Your answer here:*

## Wrap-up

- A **straight line** can't predict a probability — it leaves the [0, 1] range.
- **Logistic regression** bends that line through an **S-curve**, so the output is always a probability between 0 and 1.
- **`predict_proba`** gives *how likely*; **`predict`** gives the yes/no by cutting the probability at **0.5**.
- Scored with **accuracy** on the test set, the spam model lands around **0.99** — a clean, balanced win.

**Next session:** why **0.5**? If one kind of mistake costs more than the other, we'd move the cut. Session 11 slides the **threshold** and watches predictions flip.

*Homework: put logistic regression head-to-head with KNN on our own students, in `homework.ipynb`.*